In [32]:
import pandas as pd
import numpy as np
import pandas as pd


model_name = "Qwen3-Next-80B-A3B-Thinking-FP8"
df = pd.read_json(f"/home/snt/projects_lujun/LLMJudgeTempCausal/output/results_with_metrics/evaluation_with_metrics_Qwen__{model_name}.jsonl", lines=True)

In [37]:


group_cols = ["question_id", "judge_type", "prompt_variant", "temperature"]


def _mu(series: pd.Series) -> float:
    s = pd.to_numeric(series, errors="coerce").dropna()
    if len(s) == 0:
        return np.nan
    return float(s.mean())


def _labels_with_parse_fallback(group: pd.DataFrame) -> list[str]:
    ordered = group.sort_values("repeat_id").reset_index(drop=True)
    labels = []
    for position, row in enumerate(ordered.itertuples(index=False)):
        pred = getattr(row, "predicted_winner")
        if pd.notna(pred) and str(pred).strip() != "":
            labels.append(str(pred).strip())
        else:
            repeat_value = getattr(row, "repeat_id", position)
            labels.append(f"__parse_fail__{repeat_value}_{position}")
    return labels


def _one_flip_consistency(labels: list[str]) -> float:
    if len(labels) <= 1:
        return 1.0
    flips = sum(labels[i] != labels[i - 1] for i in range(1, len(labels)))
    return 1.0 - flips / (len(labels) - 1)


missing = [c for c in group_cols + ["agreement"] if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

if "predicted_winner" not in df.columns:
    raise ValueError("Column 'predicted_winner' is required to compute consistency.")

error_col = "error_rate" if "error_rate" in df.columns else "format_error"
if error_col not in df.columns:
    raise ValueError("Need either 'error_rate' or 'format_error' column.")

if "repeat_id" not in df.columns:
    df = df.copy()
    df["repeat_id"] = np.arange(len(df))

rows = []
for keys, g in df.groupby(group_cols, dropna=False):
    key_dict = dict(zip(group_cols, keys))

    a_series = pd.to_numeric(g["agreement"], errors="coerce")
    e_series = pd.to_numeric(g[error_col], errors="coerce")

    seq = _labels_with_parse_fallback(g)
    c_val = _one_flip_consistency(seq)

    rows.append(
        {
            **key_dict,
            "n_rows": int(len(g)),
            "Agreement (µ)": _mu(a_series),
            "Consistency (µ)": c_val,
            "Error Rate (µ)": _mu(e_series),
        }
    )

metrics_by_group = (
    pd.DataFrame(rows)
    .sort_values(group_cols)
    .reset_index(drop=True)
)

print(f"Computed groups: {len(metrics_by_group)}")
print("Empty consistency groups:", int(metrics_by_group["Consistency (µ)"].isna().sum()))
display(metrics_by_group.head(20))



Computed groups: 18000
Empty consistency groups: 0


,question_id,judge_type,prompt_variant,temperature,n_rows,Agreement (µ),Consistency (µ),Error Rate (µ)
0,0,pairwise,baseline,0.01,10,1.0,0.0,0.9
1,0,pairwise,baseline,0.50,10,1.0,0.0,0.9
2,0,pairwise,baseline,1.00,10,1.0,0.0,0.9
3,0,pairwise,baseline,1.50,10,1.0,0.0,0.9
4,0,pairwise,baseline,2.00,10,1.0,0.0,0.9
5,0,pairwise,baseline,3.00,10,1.0,0.0,0.9
6,0,pairwise,cot,0.01,10,1.0,0.0,0.9
7,0,pairwise,cot,0.50,10,1.0,0.0,0.9
8,0,pairwise,cot,1.00,10,1.0,0.0,0.9
9,0,pairwise,cot,1.50,10,1.0,0.0,0.9


In [38]:

save_path = f"/home/snt/projects_lujun/LLMJudgeTempCausal/output/result_without_rep_id/metrics_by_question_judge_prompt_temp_{model_name}.jsonl"
metrics_by_group.to_json(save_path, orient="records", lines=True)
print("Saved:", save_path)

Saved: /home/snt/projects_lujun/LLMJudgeTempCausal/output/result_without_rep_id/metrics_by_question_judge_prompt_temp_Qwen3-Next-80B-A3B-Thinking-FP8.jsonl
